In [3]:
from dotenv import load_dotenv
from huggingface_hub import login
import torch
from datasets import load_dataset
from tqdm import tqdm
import os

# Silence warnings and clean up output
import transformers
from transformers import pipeline
transformers.logging.set_verbosity_error() 
transformers.utils.logging.disable_progress_bar()
import warnings, logging

from huggingface_hub.utils import disable_progress_bars
disable_progress_bars() 

# PEFT/ Lora Imports
from transformers import LlavaForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model

load_dotenv()

login(token=os.getenv("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


cuda


In [ ]:
model_id = "llava-hf/llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto"
)

config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj)",
    use_4bit=True,
)
model = get_peft_model(model, config)
model.print_trainable_parameters()  

trainable params: 16,777,216 || all params: 7,080,204,288 || trainable%: 0.2370


In [ ]:
# CUB-200 Birds dataset

pipe = pipeline(
    "image-text-to-text",
    model="llava-hf/llava-1.5-7b-hf",
    device=device,
    dtype=torch.float16,
)

dataset = load_dataset("cassiekang/cub200_dataset")
print(dataset)
print(dataset["test"].features) 

def make_messages(img):
    return [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "Identify the bird in the photo"},
        ],
    }]

def run(row):
    return pipe(text=make_messages(row["image"]), max_new_tokens=60,
                return_full_text=False)[0]["generated_text"]

run(dataset["test"][0])

for i in tqdm(range(3)):
    row = dataset["train"][i]
    pred = run(row)
    tqdm.write(f"[{i}] Pred:  {pred.strip()}\n[{i}] Truth: {row['text']}\n")

DatasetDict({
    train: Dataset({
        features: ['text', 'image'],
        num_rows: 5994
    })
    test: Dataset({
        features: ['text', 'image'],
        num_rows: 5794
    })
})
{'text': Value('string'), 'image': Image(mode=None, decode=True)}
{'input_ids': (1, 613), 'attention_mask': (1, 613), 'pixel_values': (1, 3, 336, 336), 'labels': (1, 613)}
Supervised answer tokens: 11
